In [1]:
%pip install -q pystac-client planetary-computer rasterio pyproj pyarrow pandas numpy tqdm "zarr>=3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

import rasterio
from rasterio.windows import Window

import pystac_client
import planetary_computer

from pyproj import CRS, Transformer
import zarr
from zarr.codecs import BloscCodec, BloscShuffle

from tqdm.auto import tqdm

In [3]:
OUTPUT_DIR = Path(
    "Data/Global/pipeline_stage1/modis"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODIS_DATASET_DIR = (
    OUTPUT_DIR / "evi_lst_patch_dataset"
)

MODIS_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PC_STAC_URL = (
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

MOD13_COLLECTION = "modis-13A1-061"
MOD11_COLLECTION = "modis-11A2-061"   # Terra 8-day LST composite

# Will be discovered from one test item.
MOD13_ASSET = None
MOD11_ASSET = None

# Item dates are the composite START dates, and an item is used for a
# sample when start <= sample_date <= start + window.
# EVI (MOD13A1) is a 16-day composite; LST (MOD11A2) is an 8-day
# composite, so a 7-day window picks the single composite that
# contains the sample date. Use 15 to median over two composites
# (better gap filling, wider time span).
EVI_WINDOW_DAYS = 30
LST_WINDOW_DAYS = 7

# Native patch sizes.
# EVI is 500 m, so 27x27 ~= 13.5 km.
# LST is 1 km, so 13x13 ~= 13 km.
EVI_PATCH_SIZE = 27
LST_PATCH_SIZE = 13

# MODIS scale factors.
EVI_SCALE = 0.0001
LST_SCALE = 0.02

# Raw validity ranges.
EVI_RAW_MIN = -2000
EVI_RAW_MAX = 10000

LST_RAW_MIN = 7500
LST_RAW_MAX = 20000

# Compact storage fill values.
EVI_FILL = np.int16(-32768)
LST_FILL = np.uint16(0)

print(
    "Output:",
    MODIS_DATASET_DIR,
)

Output: Data/Global/pipeline_stage1/modis/evi_lst_patch_dataset


In [4]:
if "samples" not in globals():
    samples = pd.read_parquet(
        "Data/Global/pipeline_stage1/station_date_candidates.parquet"
    )

if "stations" not in globals():
    stations = pd.read_parquet(
        "Data/Global/pipeline_stage1/stations.parquet"
    )

samples["sample_date"] = (
    pd.to_datetime(
        samples["sample_date"]
    )
    .dt.normalize()
)

samples["station_id"] = (
    samples["station_id"]
    .astype(str)
)

stations["station_id"] = (
    stations["station_id"]
    .astype(str)
)

print(
    "Samples:",
    f"{len(samples):,}",
)

print(
    "Stations:",
    f"{len(stations):,}",
)

Samples: 3,514,643
Stations: 2,210


In [5]:
catalog = pystac_client.Client.open(
    PC_STAC_URL,
    modifier=planetary_computer.sign_inplace,
)

print(
    "Planetary Computer connected."
)

Planetary Computer connected.


In [6]:
MODIS_SINU_CRS = CRS.from_proj4(
    "+proj=sinu "
    "+R=6371007.181 "
    "+nadgrids=@null "
    "+wktext"
)

WGS84 = CRS.from_epsg(4326)

lonlat_to_modis = Transformer.from_crs(
    WGS84,
    MODIS_SINU_CRS,
    always_xy=True,
)

MODIS_X_ORIGIN = 20015109.354   # half the equatorial circumference
MODIS_Y_ORIGIN = 10007554.677   # half the meridian length (grid is 36 x 18 tiles)
MODIS_TILE_SIZE = 1111950.5196666666


def modis_tile_from_xy(x, y):

    h = np.floor(
        (x + MODIS_X_ORIGIN)
        / MODIS_TILE_SIZE
    ).astype(int)

    v = np.floor(
        (MODIS_Y_ORIGIN - y)
        / MODIS_TILE_SIZE
    ).astype(int)

    return h, v

In [7]:
station_x, station_y = (
    lonlat_to_modis.transform(
        stations["longitude"]
        .to_numpy(dtype=float),

        stations["latitude"]
        .to_numpy(dtype=float),
    )
)

modis_h, modis_v = (
    modis_tile_from_xy(
        station_x,
        station_y,
    )
)

stations_modis = stations.copy()

stations_modis["modis_x"] = station_x
stations_modis["modis_y"] = station_y

stations_modis["modis_h"] = modis_h
stations_modis["modis_v"] = modis_v

stations_modis["modis_tile"] = (
    "h"
    + stations_modis["modis_h"].map(
        lambda x: f"{x:02d}"
    )
    + "v"
    + stations_modis["modis_v"].map(
        lambda x: f"{x:02d}"
    )
)

print(
    "Unique MODIS tiles:",
    stations_modis[
        "modis_tile"
    ].nunique(),
)

display(
    stations_modis[
        [
            "station_id",
            "latitude",
            "longitude",
            "modis_tile",
        ]
    ].head()
)

Unique MODIS tiles: 73


,station_id,latitude,longitude,modis_tile
0,AMMA-CATCH__Banizoumbou,13.53250,2.66040,h18v07
1,AMMA-CATCH__Belefoungou-Mid,9.79506,1.70994,h18v08
2,AMMA-CATCH__Belefoungou-Top,9.78986,1.70994,h18v08
3,AMMA-CATCH__Nalohou-Mid,9.74530,1.60530,h18v08
4,AMMA-CATCH__Nalohou-Top,9.74407,1.60580,h18v08


In [8]:
station_lookup = (
    stations_modis
    .set_index(
        "station_id",
        verify_integrity=True,
    )
)

print(
    "Station lookup ready."
)

Station lookup ready.


In [9]:
def find_asset_key(
    item,
    candidates,
):

    names = list(
        item.assets.keys()
    )

    lowered = {
        name.lower(): name
        for name in names
    }

    for candidate in candidates:

        if candidate.lower() in lowered:
            return lowered[
                candidate.lower()
            ]

    for candidate in candidates:

        for name in names:

            if (
                candidate.lower()
                in name.lower()
            ):
                return name

    raise KeyError(
        f"Could not find asset {candidates}. "
        f"Available assets: {names}"
    )

In [10]:
TEST_STATION = stations_modis.iloc[0]

TEST_LON = float(
    TEST_STATION["longitude"]
)

TEST_LAT = float(
    TEST_STATION["latitude"]
)

TEST_TILE = (
    TEST_STATION["modis_tile"]
)

TEST_DATE = (
    samples["sample_date"]
    .sort_values()
    .iloc[
        len(samples) // 2
    ]
)

print(
    "Station:",
    TEST_STATION["station_id"],
)

print(
    "Tile:",
    TEST_TILE,
)

print(
    "Date:",
    TEST_DATE,
)

Station: AMMA-CATCH__Banizoumbou
Tile: h18v07
Date: 2020-01-24 00:00:00


In [11]:
search = catalog.search(
    collections=[
        MOD13_COLLECTION
    ],
    bbox=[
        TEST_LON - 0.2,
        TEST_LAT - 0.2,
        TEST_LON + 0.2,
        TEST_LAT + 0.2,
    ],
    datetime=(
        (
            TEST_DATE
            - pd.Timedelta(
                days=EVI_WINDOW_DAYS
            )
        ).isoformat(),

        (
            TEST_DATE
            + pd.Timedelta(days=1)
        ).isoformat(),
    ),
)

mod13_items = [
    item
    for item in search.items()
    if TEST_TILE.lower()
    in item.id.lower()
]

if not mod13_items:
    raise RuntimeError(
        "No MOD13A1 item found."
    )

MOD13_ASSET = find_asset_key(
    mod13_items[0],
    [
        "500m_16_days_EVI",
        "EVI",
    ],
)

print(
    "MOD13 asset:",
    MOD13_ASSET,
)

print(
    "Example:",
    mod13_items[0].id,
)

MOD13 asset: 500m_16_days_EVI
Example: MYD13A1.A2020025.h18v07.061.2020328195254


In [12]:
search = catalog.search(
    collections=[
        MOD11_COLLECTION
    ],
    bbox=[
        TEST_LON - 0.2,
        TEST_LAT - 0.2,
        TEST_LON + 0.2,
        TEST_LAT + 0.2,
    ],
    datetime=(
        (
            TEST_DATE
            - pd.Timedelta(
                days=LST_WINDOW_DAYS
            )
        ).isoformat(),

        (
            TEST_DATE
            + pd.Timedelta(days=1)
        ).isoformat(),
    ),
)

mod11_items = [
    item
    for item in search.items()
    if TEST_TILE.lower()
    in item.id.lower()
]

if not mod11_items:
    raise RuntimeError(
        "No MOD11A2 item found."
    )

MOD11_ASSET = find_asset_key(
    mod11_items[0],
    [
        "LST_Day_1km",
        "LST_Day",
    ],
)

print(
    "MOD11 asset:",
    MOD11_ASSET,
)

print(
    "Example:",
    mod11_items[0].id,
)

MOD11 asset: LST_Day_1km
Example: MYD11A2.A2020025.h18v07.061.2021005134152


In [13]:
os.environ[
    "GDAL_DISABLE_READDIR_ON_OPEN"
] = "EMPTY_DIR"

os.environ[
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS"
] = ".tif,.TIF"

os.environ[
    "GDAL_HTTP_MULTIRANGE"
] = "YES"

os.environ[
    "VSI_CACHE"
] = "TRUE"

os.environ[
    "VSI_CACHE_SIZE"
] = str(
    64 * 1024 * 1024
)

In [14]:
def read_native_patch(
    href,
    x,
    y,
    patch_size,
    valid_min,
    valid_max,
):

    radius = patch_size // 2

    with rasterio.open(
        href
    ) as src:

        row, col = src.index(
            x,
            y,
        )

        window = Window(
            col - radius,
            row - radius,
            patch_size,
            patch_size,
        )

        patch = src.read(
            1,
            window=window,
            boundless=True,
            fill_value=0,
        ).astype(
            np.float32
        )

    valid = (
        np.isfinite(patch)
        &
        (patch >= valid_min)
        &
        (patch <= valid_max)
    )

    patch[~valid] = np.nan

    return patch

In [15]:
item = planetary_computer.sign(
    mod13_items[-1]
)

evi_href = item.assets[
    MOD13_ASSET
].href

evi_test = read_native_patch(
    evi_href,
    TEST_STATION["modis_x"],
    TEST_STATION["modis_y"],
    EVI_PATCH_SIZE,
    EVI_RAW_MIN,
    EVI_RAW_MAX,
)

valid = np.isfinite(
    evi_test
)

print(
    "EVI shape:",
    evi_test.shape,
)

print(
    "Valid:",
    valid.sum(),
    "/",
    valid.size,
)

print(
    "Physical range:",
    (
        evi_test[valid]
        * EVI_SCALE
    ).min(),
    (
        evi_test[valid]
        * EVI_SCALE
    ).max(),
)

EVI shape: (27, 27)
Valid: 729 / 729
Physical range: 0.0824 0.1785


In [16]:
item = planetary_computer.sign(
    mod11_items[-1]
)

lst_href = item.assets[
    MOD11_ASSET
].href

lst_test = read_native_patch(
    lst_href,
    TEST_STATION["modis_x"],
    TEST_STATION["modis_y"],
    LST_PATCH_SIZE,
    LST_RAW_MIN,
    LST_RAW_MAX,
)

valid = np.isfinite(
    lst_test
)

print(
    "LST shape:",
    lst_test.shape,
)

print(
    "Valid:",
    valid.sum(),
    "/",
    valid.size,
)

print(
    "Physical range K:",
    (
        lst_test[valid]
        * LST_SCALE
    ).min(),
    (
        lst_test[valid]
        * LST_SCALE
    ).max(),
)

LST shape: (13, 13)
Valid: 169 / 169
Physical range K: 303.1 305.34


In [17]:
samples_modis = (
    samples[
        [
            "sample_id",
            "station_id",
            "sample_date",
        ]
    ]
    .copy()
)

samples_modis = (
    samples_modis
    .merge(
        stations_modis[
            [
                "station_id",
                "modis_tile",
                "modis_x",
                "modis_y",
            ]
        ],
        on="station_id",
        how="left",
        validate="many_to_one",
    )
)

print(
    "MODIS samples:",
    f"{len(samples_modis):,}",
)

MODIS samples: 3,514,643


In [18]:
import re

_DOY = re.compile(r"\.A(\d{4})(\d{3})\.")


def item_date(item):
    """Acquisition/composite start date as a tz-naive Timestamp.

    MODIS items on Planetary Computer can have datetime = None
    (only start_datetime/end_datetime), so parse the AYYYYDDD
    token in the item id first.
    """

    m = _DOY.search(item.id)

    if m:
        return (
            pd.Timestamp(
                year=int(m.group(1)),
                month=1,
                day=1,
            )
            + pd.Timedelta(
                days=int(m.group(2)) - 1
            )
        )

    dt = (
        item.datetime
        or item.properties.get("start_datetime")
    )

    if dt is None:
        return None

    ts = pd.Timestamp(dt)

    if ts.tzinfo is not None:
        ts = ts.tz_convert(None)

    return ts.normalize()


def _search_tile_product(
    collection_id,
    tile,
    tile_stations,
    start_date,
    end_date,
):

    search = catalog.search(
        collections=[
            collection_id
        ],
        bbox=[
            tile_stations[
                "longitude"
            ].min() - 0.1,

            tile_stations[
                "latitude"
            ].min() - 0.1,

            tile_stations[
                "longitude"
            ].max() + 0.1,

            tile_stations[
                "latitude"
            ].max() + 0.1,
        ],
        datetime=(
            start_date.isoformat(),
            end_date.isoformat(),
        ),
    )

    result = {}

    for item in search.items():

        if (
            tile.lower()
            not in item.id.lower()
        ):
            continue

        d = item_date(item)

        if d is None:
            continue

        result[d] = item

    return result


import threading

_SEARCH_LOCK = threading.Lock()


def search_tile_product(*args, **kwargs):
    # pystac-client's HTTP session is shared; serialise searches so
    # several years can run in threads safely. Searches are cheap.
    with _SEARCH_LOCK:
        return _search_tile_product(*args, **kwargs)


In [19]:
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

GDAL_OPTS = dict(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    GDAL_HTTP_MAX_RETRY="5",
    GDAL_HTTP_RETRY_DELAY="1",
    VSI_CACHE="TRUE",
    GDAL_HTTP_TIMEOUT="90",
    GDAL_HTTP_CONNECTTIMEOUT="30",
)


def read_window_safe(src, row, col, size, vmin, vmax):
    """Read a size x size window centred on (row, col).

    Pixels outside the raster stay NaN (no boundless/VRT read,
    and no fake zeros that would count as valid EVI).
    """

    r0 = row - size // 2
    c0 = col - size // 2

    out = np.full(
        (size, size), np.nan, dtype=np.float32
    )

    rr0 = max(r0, 0)
    cc0 = max(c0, 0)
    rr1 = min(r0 + size, src.height)
    cc1 = min(c0 + size, src.width)

    if rr1 <= rr0 or cc1 <= cc0:
        return out

    data = src.read(
        1,
        window=Window(
            cc0, rr0, cc1 - cc0, rr1 - rr0
        ),
    ).astype(np.float32)

    data[(data < vmin) | (data > vmax)] = np.nan

    out[
        rr0 - r0 : rr1 - r0,
        cc0 - c0 : cc1 - c0,
    ] = data

    return out


# Below this many needed stations in a tile, small window reads beat
# downloading the whole band. Tune after timing one tile.
FULL_BAND_MIN_STATIONS = 25


def fetch_item_patches(
    item,
    asset_key,
    xs,
    ys,
    patch_size,
    vmin,
    vmax,
    need=None,
    attempts=8,
):
    """Read patches from one item for the stations flagged in `need`.

    Returns float32 (n_stations, patch_size, patch_size); rows for
    stations that are not needed stay NaN. Uses one whole-band
    download when many stations are needed, otherwise per-station
    window reads. Retries transient errors and raises if they persist.
    """

    xs = np.asarray(xs)
    ys = np.asarray(ys)

    if need is None:
        need = np.ones(len(xs), dtype=bool)

    sel = np.flatnonzero(need)
    radius = patch_size // 2

    for attempt in range(attempts):

        out = np.full(
            (len(xs), patch_size, patch_size),
            np.nan,
            dtype=np.float32,
        )

        if sel.size == 0:
            return out

        try:

            signed = planetary_computer.sign(item)
            href = signed.assets[asset_key].href

            with rasterio.Env(**GDAL_OPTS):
                with rasterio.open(href) as src:

                    rows, cols = rasterio.transform.rowcol(
                        src.transform, xs[sel], ys[sel]
                    )
                    rows = np.asarray(rows, dtype=np.int64)
                    cols = np.asarray(cols, dtype=np.int64)

                    if sel.size >= FULL_BAND_MIN_STATIONS:

                        data = src.read(1).astype(np.float32)
                        data[
                            (data < vmin) | (data > vmax)
                        ] = np.nan

                        height, width = data.shape

                        padded = np.pad(
                            data,
                            radius,
                            constant_values=np.nan,
                        )
                        del data

                        for k, (r, c) in enumerate(
                            zip(rows, cols)
                        ):
                            if 0 <= r < height and 0 <= c < width:
                                out[sel[k]] = padded[
                                    r : r + patch_size,
                                    c : c + patch_size,
                                ]

                    else:

                        for k, (r, c) in enumerate(
                            zip(rows, cols)
                        ):
                            out[sel[k]] = read_window_safe(
                                src,
                                int(r),
                                int(c),
                                patch_size,
                                vmin,
                                vmax,
                            )

            return out

        except Exception:

            if attempt == attempts - 1:
                raise

            time.sleep(min(2 ** (attempt + 1), 60))


In [20]:
def encode_evi(
    patch
):

    out = np.full(
        patch.shape,
        EVI_FILL,
        dtype=np.int16,
    )

    valid = np.isfinite(
        patch
    )

    out[valid] = np.rint(
        patch[valid]
    ).astype(
        np.int16
    )

    return out


def encode_lst(
    patch
):

    out = np.full(
        patch.shape,
        LST_FILL,
        dtype=np.uint16,
    )

    valid = np.isfinite(
        patch
    )

    out[valid] = np.rint(
        patch[valid]
    ).astype(
        np.uint16
    )

    return out

In [21]:
COMPRESSOR = BloscCodec(
    cname="zstd",
    clevel=5,
    shuffle=BloscShuffle.bitshuffle,
)


def create_year_store(
    year,
    year_samples,
):

    year_dir = (
        MODIS_DATASET_DIR
        / str(year)
    )

    year_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    zarr_path = (
        year_dir
        / "patches.zarr"
    )

    group = zarr.open_group(
        str(zarr_path),
        mode="a",
    )

    n = len(
        year_samples
    )

    if "evi" not in group:

        group.create_array(
            name="evi",
            shape=(
                n,
                EVI_PATCH_SIZE,
                EVI_PATCH_SIZE,
            ),
            chunks=(
                128,
                EVI_PATCH_SIZE,
                EVI_PATCH_SIZE,
            ),
            dtype="int16",
            fill_value=int(
                EVI_FILL
            ),
            compressors=COMPRESSOR,
        )

        group.create_array(
            name="lst",
            shape=(
                n,
                LST_PATCH_SIZE,
                LST_PATCH_SIZE,
            ),
            chunks=(
                256,
                LST_PATCH_SIZE,
                LST_PATCH_SIZE,
            ),
            dtype="uint16",
            fill_value=int(
                LST_FILL
            ),
            compressors=COMPRESSOR,
        )

        group.create_array(
            name="done",
            shape=(n,),
            chunks=(4096,),
            dtype="bool",
            fill_value=False,
            compressors=COMPRESSOR,
        )

    metadata_path = (
        year_dir
        / "samples.parquet"
    )

    if not metadata_path.exists():

        year_samples[
            [
                "sample_id",
                "station_id",
                "sample_date",
                "modis_tile",
            ]
        ].to_parquet(
            metadata_path,
            index=False,
        )

    return group

/tmp/ipykernel_26969/833500336.py:4: DeprecationWarning: BloscShuffle.bitshuffle is deprecated; pass the string 'bitshuffle' instead.
  shuffle=BloscShuffle.bitshuffle,


In [22]:
def process_modis_year(
    year,
    max_workers=8,
    chunk_dates=30,
):

    year_start = pd.Timestamp(f"{year}-01-01")
    year_end = pd.Timestamp(f"{year + 1}-01-01")

    year_samples = (
        samples_modis[
            (samples_modis["sample_date"] >= year_start)
            & (samples_modis["sample_date"] < year_end)
        ]
        .sort_values(["sample_date", "sample_id"])
        .reset_index(drop=True)
    )

    if year_samples.empty:
        print(f"{year}: no samples")
        return

    group = create_year_store(year, year_samples)

    # Resume state held in memory; written back per chunk.
    done_arr = group["done"][:].astype(bool)

    for tile, tile_idx in tqdm(
        year_samples.groupby("modis_tile").groups.items(),
        desc=f"{year} tiles",
    ):

        tile_sample = year_samples.loc[tile_idx].copy()

        pending = tile_sample[
            ~done_arr[tile_sample.index.to_numpy()]
        ].sort_values("sample_date")

        if pending.empty:
            continue

        tile_stations = (
            station_lookup
            .loc[tile_sample["station_id"].unique()]
            .reset_index()
        )

        st_pos = {
            sid: i
            for i, sid in enumerate(
                tile_stations["station_id"]
            )
        }

        xs = tile_stations["modis_x"].to_numpy()
        ys = tile_stations["modis_y"].to_numpy()

        # Stations are fixed but many only exist for part of the
        # period: an item is read only for stations that have a
        # pending sample whose window contains the item's date.
        s_dates = pending["sample_date"].to_numpy()
        s_pos = np.array(
            [st_pos[s] for s in pending["station_id"]],
            dtype=np.int64,
        )

        def need_mask(d, window):
            lo = np.searchsorted(
                s_dates, d.to_datetime64(), side="left"
            )
            hi = np.searchsorted(
                s_dates,
                (d + window).to_datetime64(),
                side="right",
            )
            m = np.zeros(len(xs), dtype=bool)
            m[s_pos[lo:hi]] = True
            return m

        search_start = pending["sample_date"].min()
        search_end = (
            pending["sample_date"].max()
            + pd.Timedelta(days=1)
        )

        products = {
            "evi": dict(
                items=search_tile_product(
                    MOD13_COLLECTION,
                    tile,
                    tile_stations,
                    search_start
                    - pd.Timedelta(days=EVI_WINDOW_DAYS),
                    search_end,
                ),
                asset=MOD13_ASSET,
                size=EVI_PATCH_SIZE,
                vmin=EVI_RAW_MIN,
                vmax=EVI_RAW_MAX,
                window=pd.Timedelta(days=EVI_WINDOW_DAYS),
            ),
            "lst": dict(
                items=search_tile_product(
                    MOD11_COLLECTION,
                    tile,
                    tile_stations,
                    search_start
                    - pd.Timedelta(days=LST_WINDOW_DAYS),
                    search_end,
                ),
                asset=MOD11_ASSET,
                size=LST_PATCH_SIZE,
                vmin=LST_RAW_MIN,
                vmax=LST_RAW_MAX,
                window=pd.Timedelta(days=LST_WINDOW_DAYS),
            ),
        }

        cache = {"evi": {}, "lst": {}}

        by_date = {
            pd.Timestamp(k): v
            for k, v in pending
            .groupby("sample_date")
            .groups.items()
        }

        dates = sorted(by_date)
        n_no_items = {"evi": 0, "lst": 0}

        for k in tqdm(
            range(0, len(dates), chunk_dates),
            leave=False,
            desc=f"{tile}",
        ):

            chunk = dates[k : k + chunk_dates]
            lo, hi = chunk[0], chunk[-1]

            # Evict cached items no longer inside any window.
            for name, cfg in products.items():
                for d in list(cache[name]):
                    if d < lo - cfg["window"]:
                        del cache[name][d]

            # Fetch every missing item for this chunk in parallel.
            jobs = {}

            with ThreadPoolExecutor(max_workers) as ex:

                for name, cfg in products.items():
                    for d, item in cfg["items"].items():
                        if (
                            lo - cfg["window"] <= d <= hi
                            and d not in cache[name]
                        ):
                            need = need_mask(d, cfg["window"])

                            if not need.any():
                                continue

                            fut = ex.submit(
                                fetch_item_patches,
                                item,
                                cfg["asset"],
                                xs,
                                ys,
                                cfg["size"],
                                cfg["vmin"],
                                cfg["vmax"],
                                need,
                            )
                            jobs[fut] = (name, d)

                for fut in as_completed(jobs):
                    name, d = jobs[fut]
                    cache[name][d] = fut.result()

            # Composite per date from the in-memory cache.
            all_idx, all_evi, all_lst = [], [], []

            for date in chunk:

                date_idx = by_date[date]

                idx = np.asarray(
                    list(date_idx), dtype=np.int64
                )

                sid = (
                    tile_sample.loc[date_idx, "station_id"]
                    .to_numpy()
                )

                pos = np.array([st_pos[s] for s in sid])

                result = {}

                for name, cfg in products.items():

                    arrs = [
                        arr[pos]
                        for d, arr in cache[name].items()
                        if date - cfg["window"] <= d <= date
                    ]

                    if arrs:
                        with warnings.catch_warnings():
                            warnings.simplefilter(
                                "ignore", RuntimeWarning
                            )
                            result[name] = np.nanmedian(
                                np.stack(arrs), axis=0
                            )
                    else:
                        n_no_items[name] += 1
                        result[name] = np.full(
                            (len(pos), cfg["size"], cfg["size"]),
                            np.nan,
                            dtype=np.float32,
                        )

                all_idx.append(idx)

                all_evi.append(
                    np.where(
                        np.isfinite(result["evi"]),
                        np.rint(result["evi"]),
                        EVI_FILL,
                    ).astype(np.int16)
                )

                all_lst.append(
                    np.where(
                        np.isfinite(result["lst"]),
                        np.rint(result["lst"]),
                        LST_FILL,
                    ).astype(np.uint16)
                )

            all_idx = np.concatenate(all_idx)
            all_evi = np.concatenate(all_evi)
            all_lst = np.concatenate(all_lst)

            order = np.argsort(all_idx)
            all_idx = all_idx[order]

            # One write per chunk instead of one per date.
            group["evi"].oindex[all_idx] = all_evi[order]
            group["lst"].oindex[all_idx] = all_lst[order]

            # Mark complete only after both arrays are written.
            group["done"].oindex[all_idx] = True
            done_arr[all_idx] = True

        if any(n_no_items.values()):
            tqdm.write(
                f"WARNING {tile}: dates with no source items "
                f"-> EVI {n_no_items['evi']}, "
                f"LST {n_no_items['lst']}"
            )

    print(f"Finished {year}")


In [23]:
def run_year(year, max_restarts=50, wait=30, **kwargs):
    """process_modis_year with automatic resume after failures.

    Progress is saved per chunk (the `done` flags), so each
    restart continues where the last one stopped.
    """

    for attempt in range(max_restarts):

        try:
            process_modis_year(year, **kwargs)
            return

        except KeyboardInterrupt:
            raise

        except Exception as e:
            print(
                f"[{year}] attempt {attempt + 1} failed: "
                f"{type(e).__name__}: {e}"
            )
            print(f"Resuming from saved progress in {wait}s ...")
            time.sleep(wait)

    raise RuntimeError(
        f"{year}: still failing after {max_restarts} restarts"
    )


def run_years_parallel(years, year_workers=3, inner_workers=6):
    """Run several years at once in threads (downloads are I/O bound).

    Each year has its own store, so they never touch each other.
    Total connections ~= year_workers * inner_workers.
    """

    failed = []

    with ThreadPoolExecutor(year_workers) as ex:

        futs = {
            ex.submit(
                run_year, int(y), max_workers=inner_workers
            ): int(y)
            for y in years
        }

        for f in as_completed(futs):
            y = futs[f]
            try:
                f.result()
                print(f"[{y}] complete")
            except Exception as e:
                failed.append(y)
                print(f"[{y}] FAILED: {type(e).__name__}: {e}")

    if failed:
        print("Failed years (rerun to resume):", sorted(failed))


TEST_YEAR = 2020

run_year(
    TEST_YEAR
)


2020 tiles:   0%|          | 0/46 [00:00<?, ?it/s]

Finished 2020


In [24]:
year_dir = (
    MODIS_DATASET_DIR
    / str(TEST_YEAR)
)

group = zarr.open_group(
    str(
        year_dir
        / "patches.zarr"
    ),
    mode="r",
)

print(
    "EVI:",
    group["evi"].shape,
)

print(
    "LST:",
    group["lst"].shape,
)

print(
    "Completed:",
    int(
        group["done"][:].sum()
    ),
    "/",
    group["done"].shape[0],
)

EVI: (269853, 27, 27)
LST: (269853, 13, 13)
Completed: 269853 / 269853


In [25]:
meta = pd.read_parquet(
    year_dir
    / "samples.parquet"
)

done = group[
    "done"
][:]

idx = np.flatnonzero(
    done
)

if len(idx) == 0:
    raise RuntimeError(
        "No completed MODIS samples."
    )

i = idx[0]

evi = (
    group["evi"][i]
    .astype(np.float32)
    * EVI_SCALE
)

lst = (
    group["lst"][i]
    .astype(np.float32)
    * LST_SCALE
)

evi[
    group["evi"][i]
    == EVI_FILL
] = np.nan

lst[
    group["lst"][i]
    == LST_FILL
] = np.nan

print(
    meta.iloc[i]
)

print(
    "\nEVI:",
    np.nanmin(evi),
    "→",
    np.nanmax(evi),
)

print(
    "LST:",
    np.nanmin(lst),
    "→",
    np.nanmax(lst),
    "K",
)

sample_id                     9369
station_id            ARM__Anthony
sample_date    2020-01-01 00:00:00
modis_tile                  h10v05
Name: 0, dtype: object

EVI: 0.085 → 0.49069998
LST: 279.68 → 281.8 K


In [26]:
years = sorted(
    samples_modis[
        "sample_date"
    ]
    .dt.year
    .unique()
)

remaining = [
    int(y) for y in years if int(y) != TEST_YEAR
]

run_years_parallel(
    remaining,
    year_workers=3,
    inner_workers=6,
)


2015 tiles:   0%|          | 0/61 [00:00<?, ?it/s]

2017 tiles:   0%|          | 0/58 [00:00<?, ?it/s]

2016 tiles:   0%|          | 0/60 [00:00<?, ?it/s]

Finished 2015
[2015] complete
Finished 2017
[2017] complete
Finished 2016
[2016] complete


2018 tiles:   0%|          | 0/51 [00:00<?, ?it/s]

2021 tiles:   0%|          | 0/43 [00:00<?, ?it/s]

... (5 outputs left out)

Finished 2023
[2023] complete
Finished 2024
[2024] complete
Finished 2022
[2022] complete


2026 tiles:   0%|          | 0/30 [00:00<?, ?it/s]

2025 tiles:   0%|          | 0/34 [00:00<?, ?it/s]

Finished 2026
[2026] complete
Finished 2025
[2025] complete
